# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/NameRectified/flyrank-ml-internship/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections in order - each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read skills/README.md first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

Write the rule in plain words first. Then the reason codes it can output.

The rule in plain words: A page is worth reviewing if it gets enough search impressions but its click-through rate falls below the median for other pages at the same position tier. The more impressions the page gets and the bigger the CTR gap, the higher the priority. This captures pages where the title, meta description, or snippet may not be pulling their weight relative to the page's actual search visibility.

Reason codes this rule outputs:
- ctr_opportunity: the page has above-minimum impressions and a CTR below its position tier median. The score is impressions multiplied by the CTR gap, so pages with high volume and large gaps rank first.

Before writing the rule, I checked two signals the rule leans on. Each gets a bucket table with n, and a one-word verdict.

Correction note: the first version of this notebook built the outcome from the feature window too, requiring the same CTR gap in both windows. That put a feature inside the label. The corrected outcome uses the observed March result only, below_tier_outcome, exactly as in the w03 data contract.

In [1]:
import os, getpass, duckdb, pandas as pd, numpy as np
from pathlib import Path

env_path = Path('../../.env')
if env_path.exists():
    for line in env_path.read_text().strip().split('\n'):
        if '=' in line:
            k, v = line.split('=', 1)
            os.environ[k.strip()] = v.strip()
HF_TOKEN = os.environ.get('HF_TOKEN') or getpass.getpass('HF READ token: ')

con = duckdb.connect()
con.execute('LOAD httpfs')
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = 'hf://datasets/FlyRank/internship-warehouse'
FACT = f"read_parquet('{REL}/fact_content_daily_performance/**/*.parquet')"
DIM_CONTENT = f"read_parquet('{REL}/dim_content.parquet')"

data = con.sql(f"""
    SELECT f.content_hash_id,
           SUM(f.gsc_impressions) AS impressions_fw,
           SUM(f.gsc_clicks) AS clicks_fw,
           AVG(f.gsc_avg_position) AS avg_pos_fw,
           STDDEV_SAMP(f.gsc_avg_position) AS pos_volatility_fw,
           SUM(f.ga4_sessions) AS sessions_fw,
           SUM(f.ga4_engaged_sessions) AS engaged_sessions_fw,
           c.content_type,
           c.main_intent,
           SUM(CASE WHEN f.report_date >= '2026-03-01' AND f.report_date < '2026-04-01'
                    THEN f.gsc_impressions ELSE 0 END) AS impressions_label,
           SUM(CASE WHEN f.report_date >= '2026-03-01' AND f.report_date < '2026-04-01'
                    THEN f.gsc_clicks ELSE 0 END) AS clicks_label
    FROM {FACT} f
    JOIN {DIM_CONTENT} c ON f.content_hash_id = c.content_hash_id
    WHERE f.report_date >= '2026-01-01' AND f.report_date < '2026-04-01'
      AND f.gsc_data_available IS TRUE
    GROUP BY f.content_hash_id, c.content_type, c.main_intent
    HAVING SUM(f.gsc_impressions) >= 100
""").df()

print(f'Loaded {len(data):,} pages with complete data')

def assign_tier(pos):
    if pos <= 3:
        return 'top_3'
    if pos <= 10:
        return 'page_1'
    if pos <= 20:
        return 'striking'
    if pos <= 50:
        return 'page_3_5'
    return 'deep'

data['ctr_fw'] = data['clicks_fw'] / data['impressions_fw'] * 100
data['position_tier'] = data['avg_pos_fw'].apply(assign_tier)

tier_med = data.groupby('position_tier').apply(
    lambda g: g['clicks_fw'].sum() / g['impressions_fw'].sum() * 100
)
data['tier_median_ctr'] = data['position_tier'].map(tier_med)
data['tier_ctr_gap'] = data['tier_median_ctr'] - data['ctr_fw']

data['ctr_label'] = data['clicks_label'] / data['impressions_label'] * 100
data['gap_label'] = data['tier_median_ctr'] - data['ctr_label']

data['below_tier_eligible'] = (data['tier_ctr_gap'] > 0.1).astype(int)
data['below_tier_outcome'] = (data['gap_label'] > 0.1).astype(int)
print(f'Base rate (below tier in March, all pages): {data["below_tier_outcome"].mean():.1%}')
eligible = data[data['below_tier_eligible'] == 1]
print(f'Base rate (below tier in March, pages already below tier in feature window): {eligible["below_tier_outcome"].mean():.1%}')
print('Label check: below_tier_outcome uses only the March outcome. The feature-window gap is never inside the label.')

data['content_type'] = data['content_type'].fillna('unknown')
data['main_intent'] = data['main_intent'].fillna('unknown')
data = data.fillna(0)

print()
print('--- Signal check 1: CTR gap by position tier (CTR-fix flag link) ---')
sig1 = data.groupby('position_tier', observed=True).agg(
    n=('below_tier_outcome', 'count'),
    median_page_ctr=('ctr_fw', 'median'),
    median_ctr_gap=('tier_ctr_gap', 'median'),
    pct_below_tier=('below_tier_outcome', 'mean')
).reset_index()
sig1['pct_below_tier'] = (sig1['pct_below_tier'] * 100).round(1)
sig1.columns = ['position_tier', 'n', 'median_page_ctr_pct', 'median_ctr_gap_pp', 'pct_below_tier']
print(sig1.to_string(index=False))
print('  Legend: position_tier = position bucket (top_3<=3, page_1=4-10, striking=11-20, page_3_5=21-50, deep>50);')
print('  n = pages in tier; median_page_ctr_pct = typical page CTR in that tier (%);')
print('  median_ctr_gap_pp = typical gap between tier-median CTR and page CTR, positive = underperforming;')
print('  pct_below_tier = % of these pages below tier in March (the label positive rate).')
print('Verdict: CONFIRMED. Pages deeper in search results have lower CTR medians, and the gap identifies pages that underperform their tier. The outcome is measured in March only.')

print()
print('--- Signal check 2: Impression volume bins (quick-win flag link) ---')
data['volume_bin'] = pd.qcut(data['impressions_fw'], q=4, labels=['low', 'medium', 'high', 'very high'])
sig2 = data.groupby('volume_bin', observed=True).agg(
    n=('below_tier_outcome', 'count'),
    avg_impressions=('impressions_fw', 'mean'),
    avg_ctr_gap=('tier_ctr_gap', 'mean'),
    pct_below_tier=('below_tier_outcome', 'mean')
).reset_index()
sig2['avg_impressions'] = sig2['avg_impressions'].round(0).astype(int)
sig2['avg_ctr_gap'] = sig2['avg_ctr_gap'].round(4)
sig2['pct_below_tier'] = (sig2['pct_below_tier'] * 100).round(1)
print(sig2.to_string(index=False))
print('  Legend: volume_bin = quartile of Jan-Feb impressions (low/medium/high/very high);')
print('  n = pages in bin; avg_impressions = average impressions in bin;')
print('  avg_ctr_gap = average gap, negative = pages typically ABOVE tier-median CTR;')
print('  pct_below_tier = % of these pages below tier in March (the label positive rate).')
print('Verdict: MIXED. Low-volume pages actually have the highest rate of below-tier outcomes, while very high-volume pages show the lowest rate and near-zero average gap. Pages at very high impression volumes tend to be at strong positions where CTR is already competitive. Volume does not predict a gap - but where a gap exists, volume scales the opportunity. The rule handles this correctly by gating on gap first, then scaling by volume.')

data = data.drop(columns=['volume_bin', 'ctr_label', 'gap_label'])
print()
print('Cleaned up label-window columns. Ready for scoring.')

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Loaded 120,258 pages with complete data
Base rate (below tier in March, all pages): 57.5%
Base rate (below tier in March, pages already below tier in feature window): 88.7%
Label check: below_tier_outcome uses only the March outcome. The feature-window gap is never inside the label.

--- Signal check 1: CTR gap by position tier (CTR-fix flag link) ---
position_tier     n  median_page_ctr_pct  median_ctr_gap_pp  pct_below_tier
         deep  4178             0.000000           0.045337             0.0
       page_1 55979             0.192864           0.134336            56.4
     page_3_5 21338             0.000000           0.149286            63.5
     striking 28801             0.114460           0.175050            63.3
        top_3  9962             0.232992           0.172887            58.8
  Legend: position_tier = position bucket (top_3<=3, page_1=4-10, striking=11-20, page_3_5=21-50, deep>50);
  n = pages in tier; median_page_ctr_pct = typical page CTR in that tier (%);
  me

/var/folders/yy/t95v8nfd5mg826zr_b6pmwf00000gp/T/ipykernel_16447/4068071704.py:58: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  tier_med = data.groupby('position_tier').apply(


## 2. Build the ranked queue (writes the CSV)

Code the score, rank everything, write work/outputs/baseline_action_score.csv.

The score is: has_volume times ctr_gap times impressions_fw.
- has_volume is 1 when impressions_fw >= 500. This filters out low-traffic pages where CTR noise would dominate.
- ctr_gap is the positive part of tier_ctr_gap. Only pages below their tier median get a positive gap.
- impressions_fw scales the score by volume. A page with 10,000 impressions and a 2pp gap is a bigger opportunity than one with 200 impressions and the same gap.
- The reason code is always ctr_opportunity. The action label is review_snippet_metadata.

In [2]:
has_volume = (data['impressions_fw'] >= 500).astype(int)
ctr_gap = data['tier_ctr_gap'].clip(lower=0)

data['score'] = has_volume * ctr_gap * data['impressions_fw']
data['reason_code'] = 'ctr_opportunity'
data['action_label'] = 'review_snippet_metadata'

queue = data[data['score'] > 0].sort_values('score', ascending=False).copy()
queue['rank'] = range(1, len(queue) + 1)

csv_cols = ['rank', 'content_hash_id', 'score', 'reason_code', 'action_label',
            'impressions_fw', 'ctr_fw', 'tier_median_ctr', 'tier_ctr_gap',
            'position_tier', 'avg_pos_fw', 'content_type', 'main_intent']

out_dir = Path('../../work/outputs')
out_dir.mkdir(parents=True, exist_ok=True)
out_path = out_dir / 'baseline_action_score.csv'
queue[csv_cols].to_csv(out_path, index=False)

print(f'Wrote {len(queue):,} rows to {out_path}')
print(f'Score range: {queue["score"].min():.1f} to {queue["score"].max():.1f}')
print(f'Median score: {queue["score"].median():.1f}')

def precision_at_k(scores, labels, k):
    top = scores.nlargest(k).index if len(scores) >= k else scores.nlargest(len(scores)).index
    return labels.loc[top].mean()

base_rate = data['below_tier_outcome'].mean()
p50 = precision_at_k(data['score'], data['below_tier_outcome'], 50)
p10 = precision_at_k(data['score'], data['below_tier_outcome'], 10)
print(f'\nBase rate (below tier in March): {base_rate:.1%}')
print(f'Precision@10: {p10:.1%}')
print(f'Precision@50: {p50:.1%}')

Wrote 56,404 rows to ../../work/outputs/baseline_action_score.csv
Score range: 0.0 to 118361.8
Median score: 381.9

Base rate (below tier in March): 57.5%
Precision@10: 100.0%
Precision@50: 96.0%


## 3. Top-20 review

For each of the top 20: action, reason code, confidence note, and what would make it wrong.

Action is always review_snippet_metadata. Reason code is always ctr_opportunity. What changes across the top 20 is the page context that drives the score and the specific thing that could make the recommendation wrong.

Top 20 review notes:

1. Rank 1 - page_1 position, 362k impressions, CTR of 0.0008% versus a tier median of 0.3272%. The gap is wide and the volume is massive. Commercial intent. What could make it wrong: if the page targets commercial keywords where users compare before clicking, a lower CTR is normal for that query type.

2. Rank 2 - page_1, 332k impressions, CTR of 0.0003%. Nearly zero clicks despite strong visibility. Informational intent. What could make it wrong: if most impressions come from featured snippet or position zero placements, there may be no clickable result to improve.

3. Rank 3 - page_1, 302k impressions, CTR of 0.0132% versus 0.3272% tier median. Commercial intent. What could make it wrong: if the tier median blends branded and non-branded queries and this page ranks for non-branded terms, the expected CTR benchmark may be too high.

4. Rank 4 - striking range, 279k impressions, almost zero CTR. Informational intent. What could make it wrong: striking range pages that appear in position zero or as a rich result have no click-through opportunity; the low CTR is a feature of the SERP layout, not the snippet.

5. Rank 5 - page_1, 384k impressions, CTR of 0.1234%. Gap is 0.2038pp. Informational. What could make it wrong: if a short-term news surge drove the impression spike, CTR may recover without intervention.

6. Rank 6 - page_1, 433k impressions, CTR of 0.1480% versus 0.3272%. Transactional intent. What could make it wrong: transactional queries often carry lower CTR because searchers visit multiple sites before purchasing.

7. Rank 7 - page_1, 311k impressions, CTR of 0.0780%. Commercial intent. What could make it wrong: comparison-targeted content naturally gets lower CTR; users click around before deciding.

8. Rank 8 - top_3, 209k impressions, CTR of 0.0530% versus a top_3 median of 0.4059%. Informational. What could make it wrong: if the page ranks for definition-style or lookup queries, the expected CTR at position 3 is much lower than the tier median accounts for.

9. Rank 9 - page_1, 463k impressions, CTR of 0.1751%. Gap is 0.1521pp, the smallest gap in the top 10. Informational. What could make it wrong: the rank comes almost entirely from volume, not gap size. If impressions drop, so does the priority.

10. Rank 10 - page_1, 209k impressions, CTR of 0.0544%. Transactional intent. What could make it wrong: for transactional pages, the conversion rate matters more than CTR. A low CTR with a high conversion rate may already be working as intended.

11. Rank 11 - page_1 position, 229k impressions, CTR of 0.0789% versus a tier median of 0.3272%. Informational intent. What could make it wrong: if the snippet already answers the query, searchers may not need to click, so the low CTR is expected rather than fixable.

12. Rank 12 - page_1, 230k impressions, CTR of 0.0831% versus 0.3272% tier median. Commercial intent. What could make it wrong: if the page competes with review or comparison results that draw the first clicks, the expected CTR benchmark may be too high.

13. Rank 13 - top_3, 239k impressions, CTR of 0.1716% versus a top_3 median of 0.4059%. Informational intent. What could make it wrong: if part of the impressions come from position zero or a rich result, the clickable placement accounts for fewer clicks than the tier median assumes.

14. Rank 14 - striking range, 312k impressions, CTR of 0.1219%. Informational intent. What could make it wrong: if most impressions sit at the bottom of page one where ads and featured results suppress clicks, the low CTR may be a layout effect rather than a snippet problem.

15. Rank 15 - page_1, 208k impressions, CTR of 0.0960% versus 0.3272%. Informational intent. What could make it wrong: if a short-term trend inflated impressions, CTR may recover without intervention once the spike fades.

16. Rank 16 - page_1, 172k impressions, CTR of 0.0587%. Informational intent. What could make it wrong: if the impressions are dominated by one query with a featured snippet on top, there may be little clickable room for the page.

17. Rank 17 - top_3, 130k impressions, CTR of 0.0532% versus a top_3 median of 0.4059%. Informational intent. What could make it wrong: if the page ranks for definition-style queries where the answer is visible on the SERP, users click less regardless of the snippet.

18. Rank 18 - page_1, 155k impressions, CTR of 0.0336%. Transactional intent. What could make it wrong: for transactional pages the conversion rate matters more than CTR; the page may already be converting for a niche query, making a low CTR acceptable.

19. Rank 19 - striking range, 178k impressions, CTR of 0.0439% versus 0.2895% tier median. Informational intent. What could make it wrong: striking range pages often sit below ads and rich results, so the gap may reflect the SERP layout more than the metadata.

20. Rank 20 - striking range, 149k impressions, CTR of 0.0000% versus 0.2895% tier median. Transactional intent. What could make it wrong: zero clicks across many impressions suggests the query mix may be off; check the query mix before treating this as a snippet issue.

In [ ]:
top20 = pd.read_csv('../../work/outputs/baseline_action_score.csv')
display_cols = ['rank', 'position_tier', 'impressions_fw', 'avg_pos_fw',
                'ctr_fw', 'tier_median_ctr', 'tier_ctr_gap', 'score',
                'content_type', 'main_intent']
top20[display_cols].head(20)

## 4. Weak picks + leakage check

Which picks look wrong and why? Confirm no product flags or future windows leaked in.

In [4]:
print('--- Weak pick check ---')
weak = queue.head(15).iloc[4:6]
for _, row in weak.iterrows():
    print(f'Rank {row["rank"]}: impressions={row["impressions_fw"]:.0f}, '
          f'pos={row["avg_pos_fw"]:.1f}, gap={row["tier_ctr_gap"]:.4f}pp, '
          f'score={row["score"]:.1f}')
print()
print('Rank 5 and 6 are borderline. Their scores come more from impression volume than from gap size.')
print('If a team has limited reviewer capacity, preferring pages where the gap is large relative to volume')
print('might surface higher-confidence fixes at the cost of trading away some raw opportunity.')
print()
print('--- Leakage check ---')
used = ['impressions_fw', 'clicks_fw', 'avg_pos_fw', 'ctr_fw', 'tier_median_ctr', 'tier_ctr_gap']
print('Columns used in the score:')
for c in used:
    print(f'  {c} (feature window, available before March 1)')
print()
print('Columns NOT in the score:')
print('  No label-window columns (impressions_label, clicks_label, gap_label, below_tier_outcome)')
print('  No product decision flags (none exist in the warehouse data)')
print()
print('Leakage check passed. Every column was recorded before March 1, 2026.')

--- Weak pick check ---
Rank 5: impressions=384139, pos=4.8, gap=0.2038pp, score=78290.3
Rank 6: impressions=433095, pos=3.8, gap=0.1792pp, score=77608.7

Rank 5 and 6 are borderline. Their scores come more from impression volume than from gap size.
If a team has limited reviewer capacity, preferring pages where the gap is large relative to volume
might surface higher-confidence fixes at the cost of trading away some raw opportunity.

--- Leakage check ---
Columns used in the score:
  impressions_fw (feature window, available before March 1)
  clicks_fw (feature window, available before March 1)
  avg_pos_fw (feature window, available before March 1)
  ctr_fw (feature window, available before March 1)
  tier_median_ctr (feature window, available before March 1)
  tier_ctr_gap (feature window, available before March 1)

Columns NOT in the score:
  No label-window columns (impressions_label, clicks_label, gap_label, below_tier_outcome)
  No product decision flags (none exist in the wareh

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled - markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime -> Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` - then submit your repo URL on the card. Done.